In [10]:
"""
============================================================
BigQuant 竞赛 AI 智能赛道 - XGBoost 滚动选股因子
============================================================

【AI 技术关键环节标注】
1. 自动化特征工程：自动生成 22 项量价技术特征
2. XGBoost 模型训练：时序滚动训练，挖掘非线性规律
3. 预测概率因子化：模型上涨概率输出作为因子值

整体流程：
1. 数据获取：中证1000 A股日频数据（bqdata 原生库）
2. 特征生成：22 个技术+量能候选特征
3. Label 构造：5日后收益率 > 0（二分类）
4. 缺失处理：前向填充 + 全局均值，单日缺失率 ≤ 40%
5. XGBoost 滚动训练：250日窗口 + 60日步长，防止未来函数
6. 因子输出：严格三列 DataFrame（date, instrument, factor）
"""

import pandas as pd
import numpy as np
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import gc

warnings.filterwarnings('ignore')

# ================== 导入 BigQuant 原生 bqdata 库 ==================
try:
    import bqdata
    BQDATA_AVAILABLE = True
except ImportError:
    BQDATA_AVAILABLE = False


def main():
    """
    AI 赛道因子主入口函数
    
    AI 技术应用关键环节：
    【环节1】自动化特征工程 - 自动衍生 22 维量价技术指标
    【环节2】XGBoost 滚动训练 - 时序滚动挖掘特征与收益的非线性关系
    【环节3】概率因子输出 - 以模型预测上涨概率作为选股因子值
    """
    
    # ==================== 第一部分：数据获取 ====================
    try:
        end_date = datetime.now().date()
        start_date = end_date - timedelta(days=1500)

        # 获取中证1000成分股
        try:
            components_df = bqdata.query(
                "SELECT code FROM bq_index_components WHERE index_code='CSI1000.ZICN' ORDER BY code"
            )
            instruments = components_df['code'].tolist()
        except:
            instruments = [
                '000001.SZA', '000002.SZA', '000858.SZA', '000860.SZA', '000876.SZA',
                '000885.SZA', '000888.SZA', '000895.SZA', '000905.SZA', '000921.SZA',
                '000938.SZA', '000951.SZA', '000958.SZA', '000969.SZA', '000975.SZA',
                '001203.SZA', '001258.SZA', '001259.SZA', '001260.SZA', '001270.SZA'
            ]

        # 获取日频行情数据
        if not BQDATA_AVAILABLE:
            # 本地测试模拟数据
            dates = pd.date_range(start_date, end_date, freq='1d')
            data_list = []
            for inst in instruments[:20]:
                np.random.seed(hash(inst) % 2**32)
                closes = 100 + np.cumsum(np.random.randn(len(dates)) * 2)
                data = pd.DataFrame({
                    'open': closes + np.random.randn(len(dates)),
                    'close': closes,
                    'high': closes + abs(np.random.randn(len(dates))),
                    'low': closes - abs(np.random.randn(len(dates))),
                    'volume': np.random.randint(1000000, 10000000, len(dates)),
                    'money': np.random.randint(10000000, 100000000, len(dates))
                }, index=dates)
                data['security'] = inst
                data_list.append(data)
            daily_data = pd.concat(data_list).reset_index().set_index(['security', 'index']).rename_axis(['security', 'date'])
            instruments = instruments[:20]
        else:
            daily_data = bqdata.get_price(
                security=instruments,
                fields=['open', 'close', 'high', 'low', 'volume', 'money'],
                start_date=start_date,
                end_date=end_date,
                freq='1d',
                fill_paused=False
            )

        # 行业分类（用于缺失值补充参考）
        try:
            codes_str = ','.join([f"'{code}'" for code in instruments[:100]])
            industry_df = bqdata.query(
                f"SELECT code, sw_l1 as industry FROM bq_stock_info WHERE code IN ({codes_str})"
            )
        except:
            industry_df = None

    except Exception as e:
        raise

    # ==================== 第二部分：自动化特征工程【AI 环节1】 ====================
    feature_dict = {}

    if isinstance(daily_data.index, pd.MultiIndex):
        securities = daily_data.index.get_level_values(0).unique()
    else:
        securities = instruments

    for instrument in securities:
        try:
            if isinstance(daily_data.index, pd.MultiIndex):
                df = daily_data.loc[instrument].copy()
            else:
                df = daily_data[daily_data.get('code') == instrument].copy() if 'code' in daily_data.columns else None

            if df is None or len(df) < 30:
                continue

            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)

            # 基础价格特征
            df['returns'] = df['close'].pct_change()
            df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
            df['high_low_ratio'] = df['high'] / (df['low'] + 1e-6)
            df['close_open_ratio'] = df['close'] / (df['open'] + 1e-6)

            # 价格位置特征
            df['highest_20'] = df['high'].rolling(20).max()
            df['lowest_20'] = df['low'].rolling(20).min()
            df['position_20'] = (df['close'] - df['lowest_20']) / (df['highest_20'] - df['lowest_20'] + 1e-6)

            # 移动平均特征
            for ma_period in [5, 10, 20, 60]:
                df[f'ma_{ma_period}'] = df['close'].rolling(ma_period).mean()
            df['ma_ratio_5_20'] = df['ma_5'] / (df['ma_20'] + 1e-6)
            df['ma_ratio_10_60'] = df['ma_10'] / (df['ma_60'] + 1e-6)

            # 动量特征
            df['momentum_5'] = df['close'] - df['close'].shift(5)
            df['momentum_10'] = df['close'] - df['close'].shift(10)
            df['roc_5'] = df['close'].pct_change(5)
            df['roc_10'] = df['close'].pct_change(10)

            # RSI 指标
            delta = df['close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
            rs = gain / (loss + 1e-6)
            df['rsi_14'] = 100 - (100 / (1 + rs))

            # MACD 指标
            ema_12 = df['close'].ewm(span=12, adjust=False).mean()
            ema_26 = df['close'].ewm(span=26, adjust=False).mean()
            df['macd_line'] = ema_12 - ema_26
            df['macd_signal'] = df['macd_line'].ewm(span=9, adjust=False).mean()
            df['macd_histogram'] = df['macd_line'] - df['macd_signal']

            # 布林带
            df['sma_20'] = df['close'].rolling(20).mean()
            df['bb_std_20'] = df['close'].rolling(20).std()
            df['bb_upper'] = df['sma_20'] + 2 * df['bb_std_20']
            df['bb_lower'] = df['sma_20'] - 2 * df['bb_std_20']
            df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-6)

            # 成交量特征
            df['volume_ma_5'] = df['volume'].rolling(5).mean()
            df['volume_ratio'] = df['volume'] / (df['volume_ma_5'] + 1e-6)
            df['money_ma_5'] = df['money'].rolling(5).mean()
            df['money_ratio'] = df['money'] / (df['money_ma_5'] + 1e-6)

            # 波动率特征
            df['volatility_20'] = df['returns'].rolling(20).std()
            df['volatility_5'] = df['returns'].rolling(5).std()

            # 收益率分布特征
            df['skewness_20'] = df['returns'].rolling(20).skew()
            df['kurt_20'] = df['returns'].rolling(20).kurt()

            feature_dict[instrument] = df
        except Exception as e:
            continue

    # ==================== 第三部分：Label 构造 ====================
    for instrument in feature_dict.keys():
        df = feature_dict[instrument]
        df['future_return_5d'] = df['close'].shift(-5).pct_change(5)
        df['label'] = (df['future_return_5d'] > 0).astype(int)
        feature_dict[instrument] = df

    # ==================== 第四部分：缺失值处理 ====================
    feature_cols = [
        'returns', 'log_returns', 'high_low_ratio', 'close_open_ratio',
        'position_20', 'ma_ratio_5_20', 'ma_ratio_10_60',
        'momentum_5', 'momentum_10', 'roc_5', 'roc_10',
        'rsi_14', 'macd_line', 'macd_signal', 'macd_histogram',
        'bb_position', 'volume_ratio', 'money_ratio',
        'volatility_20', 'volatility_5', 'skewness_20', 'kurt_20'
    ]

    train_data = []
    for instrument, df in feature_dict.items():
        valid_idx = df['label'].notna()
        df_valid = df[valid_idx].copy()
        if len(df_valid) < 100:
            continue
        df_valid['instrument'] = instrument
        df_valid['date'] = df_valid.index
        train_data.append(df_valid)

    if len(train_data) == 0:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    train_df = pd.concat(train_data, ignore_index=False)

    # 多层缺失值填充
    for col in feature_cols:
        if col in train_df.columns:
            train_df[col] = train_df.groupby('instrument')[col].fillna(method='ffill')
            train_df[col] = train_df.groupby('instrument')[col].fillna(method='bfill')

    for col in feature_cols:
        if col in train_df.columns:
            col_mean = train_df[col].mean()
            train_df[col].fillna(col_mean, inplace=True)

    # 缺失率校验与过滤（≤40%）
    missing_rates = {}
    for date in train_df['date'].unique():
        date_data = train_df[train_df['date'] == date]
        missing_count = date_data[feature_cols].isna().sum().sum()
        total_count = len(date_data) * len(feature_cols)
        missing_rate = missing_count / total_count if total_count > 0 else 0
        missing_rates[date] = missing_rate

    max_missing_rate = max(missing_rates.values()) if missing_rates else 0
    if max_missing_rate > 0.40:
        valid_dates = [d for d, r in missing_rates.items() if r <= 0.40]
        train_df = train_df[train_df['date'].isin(valid_dates)]

    if len(train_df) < 1000:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ==================== 第五部分：XGBoost 滚动训练【AI 核心环节2】 ====================
    unique_dates = sorted(train_df['date'].unique())
    all_predictions = []

    train_window_size = 250
    step_size = 60
    total_iterations = max(0, (len(unique_dates) - train_window_size) // step_size)

    iteration_count = 0
    for i in range(0, len(unique_dates) - train_window_size, step_size):
        iteration_count += 1

        train_dates = unique_dates[i:i + train_window_size]
        test_start_idx = i + train_window_size
        test_end_idx = min(i + train_window_size + step_size, len(unique_dates))
        test_dates = unique_dates[test_start_idx:test_end_idx]

        if len(test_dates) == 0:
            break

        try:
            X_train = train_df[train_df['date'].isin(train_dates)][feature_cols].values
            y_train = train_df[train_df['date'].isin(train_dates)]['label'].values

            valid_mask = ~np.isnan(X_train).any(axis=1)
            X_train = X_train[valid_mask]
            y_train = y_train[valid_mask]

            if len(X_train) < 100:
                continue

            # 特征标准化
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)

            # XGBoost 分类模型
            model = XGBClassifier(
                n_estimators=80,
                max_depth=3,
                learning_rate=0.15,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=0,
                n_jobs=-1
            )
            model.fit(X_train_scaled, y_train)

            # 预测 - 上涨概率作为因子值【AI 环节3】
            test_data = train_df[train_df['date'].isin(test_dates)].copy()
            X_test = test_data[feature_cols].values

            valid_mask_test = ~np.isnan(X_test).any(axis=1)
            X_test_valid = X_test[valid_mask_test]
            test_data_valid = test_data[valid_mask_test].copy()

            if len(X_test_valid) > 0:
                X_test_scaled = scaler.transform(X_test_valid)
                pred_proba = model.predict_proba(X_test_scaled)[:, 1]
                test_data_valid['factor'] = pred_proba
                all_predictions.append(test_data_valid)

        except Exception as e:
            continue

        gc.collect()

    # ==================== 第六部分：因子输出（严格三列） ====================
    if len(all_predictions) == 0:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    factor_output = pd.concat(all_predictions, ignore_index=True)

    # 严格只保留三列：date, instrument, factor
    result = factor_output[['date', 'instrument', 'factor']].copy()
    result = result.sort_values(['date', 'instrument']).reset_index(drop=True)

    return result
